[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JanNikolaidovic/python-text-analytics-lab/blob/main/labs/lab15_visualization/lab15_student.ipynb)

# Lab 15 — Data Visualization with Matplotlib and Seaborn

### Digital Skills and Business Decisioning Using Python

In Lab 14 you summarised the reviews with tables. Tables are precise, but people don't see patterns in columns of numbers: a skewed distribution, an outlier, or a trend is obvious in a chart and nearly invisible in a table. Charts are also how you'll **communicate** your findings in the final assignment.

A good chart is not decoration. It answers one question, it uses the right form for that question, and it can be read correctly in a few seconds. This lab teaches the tools (**Matplotlib** and **Seaborn**) and, just as importantly, how to choose the chart that fits the question.

In this lab you'll learn to:

* build a chart with Matplotlib's **figure and axes**, and label it properly,
* choose the right chart for the question: **bar charts**, **histograms**, **box plots**, **scatter plots**, **heatmaps**, **stacked bars**, and **line/point plots**,
* make the same charts in one line with **Seaborn**, straight from a DataFrame,
* chart **text**: the most frequent words, overall and by sentiment,
* put several charts side by side with **subplots**, and
* avoid the common ways charts mislead.

### Setup

The same `load_reviews()` function as in Lab 14, followed by the plotting imports. Matplotlib's plotting module is always imported as `plt`, and Seaborn as `sns`. `sns.set_theme()` gives every chart a clean default style.

The last three lines define colours once, so every chart in the notebook uses them consistently: one blue for single-series charts, and a fixed colour for each sentiment (red for negative, grey for neutral, blue for positive). These particular colours were chosen so that they remain distinguishable for readers with common forms of colour blindness.

In [ ]:
import re
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/JanNikolaidovic/python-text-analytics-lab/main/data/womens_clothing_ecommerce_reviews.csv"

def load_reviews(url):
    """Load the clothing reviews and apply the cleaning steps from Lab 13."""
    reviews = pd.read_csv(url, index_col=0)
    reviews.columns = [name.lower().replace(" ", "_") for name in reviews.columns]
    reviews = reviews.rename(columns={"recommended_ind": "recommended",
                                      "division_name": "division",
                                      "department_name": "department"})
    reviews = reviews.dropna(subset=["review_text", "division", "department", "class_name"])
    reviews["title"] = reviews["title"].fillna("")
    reviews = reviews.drop_duplicates(subset=["review_text"])
    reviews["division"] = reviews["division"].replace({"Initmates": "Intimates"})
    reviews["recommended"] = reviews["recommended"].astype(bool)
    reviews["review_text"] = reviews["review_text"].str.replace(r"\s+", " ", regex=True).str.strip()

    # features from the text (Labs 09 and 13)
    reviews["n_words"] = reviews["review_text"].str.split().str.len()
    reviews["n_exclaims"] = reviews["review_text"].str.count("!")
    reviews["mentions_return"] = reviews["review_text"].str.contains(r"\breturn\w*", case=False)
    height = reviews["review_text"].str.extract(r"(\d)'\s?(\d{1,2})").astype(float)
    reviews["height_cm"] = ((height[0] * 12 + height[1]) * 2.54).round()
    reviews["sentiment"] = reviews["rating"].map({1: "negative", 2: "negative", 3: "neutral",
                                                  4: "positive", 5: "positive"})
    return reviews


df = load_reviews(DATA_URL)
print(df.shape)
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

BLUE = "#2a78d6"
ORDER = ["negative", "neutral", "positive"]
SENTIMENT_COLORS = {"negative": "#e34948", "neutral": "#85847e", "positive": "#2a78d6"}

### Part 1: Anatomy of a Matplotlib Chart

Every Matplotlib chart lives on a **figure** (the whole image) that contains one or more **axes** (a single plot area, with its x-axis and y-axis). The standard way to start is:

```python
fig, ax = plt.subplots()
```

which creates one figure with one set of axes and hands you both. You then **draw on `ax`** and **label `ax`**. Here's how many reviews have each star rating:

In [ ]:
rating_counts = df["rating"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(rating_counts.index, rating_counts.values, color=BLUE)
ax.set_title("Most reviews give 5 stars")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of reviews")
plt.show()

Line by line:

* `figsize=(7, 4)` sets the size in inches (width, height).
* `ax.bar(x, heights)` draws the bars. A `value_counts()` Series gives you both: its **index** (the ratings) for the x positions and its **values** (the counts) for the heights.
* `ax.set_title()`, `ax.set_xlabel()`, `ax.set_ylabel()` add the text. **Every chart needs all three.** A chart without axis labels forces the reader to guess.
* `plt.show()` displays the finished figure.

Notice the title. *"Most reviews give 5 stars"* states the **finding**; *"Rating counts"* would only name the data. When a chart is meant to make a point, let the title make it.

### Seaborn: The Same Chart From a DataFrame

**Seaborn** is built on top of Matplotlib and works directly with DataFrames: you pass the DataFrame as `data=` and name the columns. It does the counting or averaging for you. `sns.countplot()` counts the rows for each value, so you don't need `value_counts()` at all:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="rating", color=BLUE, ax=ax)
ax.set_title("Most reviews give 5 stars")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of reviews")
plt.show()

`ax=ax` tells Seaborn to draw on the axes you created, so you can still label them with the Matplotlib methods you just learned. This is the pattern for the rest of the lab: **create the figure with `plt.subplots()`, draw with Seaborn (or Matplotlib), label with `ax.set_...`.**

### Which Chart for Which Question?

The most important decision comes before any code: **what question is the chart answering?** The question picks the chart.

| Question | Data | Chart |
|---|---|---|
| How many in each category? | one categorical column | **bar chart** |
| How is a number distributed? | one numeric column | **histogram** |
| How does a number differ between groups? | numeric + categorical | **box plot**, or bars of the mean |
| Do two numbers move together? | two numeric columns | **scatter plot**; **heatmap** for many pairs |
| How do shares differ between groups? | two categorical columns | **stacked 100% bar chart**, or a **heatmap** of a crosstab |
| How does a value change along an ordered scale or over time? | ordered x + a number | **line** or **point plot** |

The rest of the lab takes these one at a time.

---

### Part 2: Bar Charts for Categories

For categories with **no natural order** (departments, classes), sort the bars by size, so the ranking can be read at a glance. For long category names, draw the bars **horizontally** with `ax.barh()`, so the labels don't overlap:

In [ ]:
department_counts = df["department"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(department_counts.index, department_counts.values, color=BLUE)
ax.bar_label(bars, padding=3)
ax.set_title("Tops and dresses make up most of the reviews")
ax.set_xlabel("Number of reviews")
ax.set_ylabel("")
plt.show()

`ax.bar_label(bars)` writes each bar's value at its end, which helps when exact numbers matter. (Use it selectively: numbers on 20 bars become clutter.) The y label is left empty on purpose, because the department names explain themselves.

Two bar-chart rules that are never optional:

* **Bars start at zero.** A bar's *length* is the value. If the axis starts at 5,000, a bar of 10,000 looks twice as long as it should.
* **One colour is enough** when the bars are one series. Colouring each department differently adds nothing (the labels already identify them), and it suggests a meaning that isn't there.

### Practice: Bar Charts

**Exercise 1: Reviews per Division (easy)**

Draw a **horizontal** bar chart of the number of reviews per `division`, sorted so the largest bar is at the top, in `BLUE`. Give it a title stating the finding, and an x-axis label.

In [ ]:
# TODO: horizontal bar chart of review counts per division, sorted, labelled

In [ ]:
# @solution
division_counts = df["division"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(division_counts.index, division_counts.values, color=BLUE)
ax.set_title("The General division has the most reviews")
ax.set_xlabel("Number of reviews")
plt.show()

---

### Part 3: Histograms for Distributions

A **histogram** shows how a numeric column is distributed: it splits the range into equal intervals (**bins**) and draws a bar for how many values fall into each. It's the chart version of `describe()`, and much more revealing.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=df, x="n_words", binwidth=3, color=BLUE, ax=ax)
ax.set_title("Review length piles up just under 100 words")
ax.set_xlabel("Words per review")
ax.set_ylabel("Number of reviews")
plt.show()

(`binwidth=3` makes every bar cover exactly 3 word counts; more on choosing bins below.)

`describe()` told you the mean and median are both about 60 words, which suggests a nice, symmetric distribution. The histogram tells a different story: lengths spread fairly evenly from 20 to 90 words, and then there's a **tall peak between about 90 and 105 words**, followed by a sudden drop.

That peak isn't a customer behaviour: it's the website's **500-character limit** (Lab 11). Many customers wrote as much as they were allowed to, so their reviews were cut off at the limit. About 1 in 5 reviews is at or near it. That's a finding worth reporting, because it means **review length is capped**: for the longest reviews, the length we measure is lower than what the customer would have written.

### Choosing the Bins

The width of the bins changes what you see. Too wide hides the shape; too narrow turns it into noise. There's no single right answer, so try a few. Here's the reviewers' age with three different bin widths, drawn **side by side**. `plt.subplots(1, 3)` creates one row of three axes, returned as a list, `axes`:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=False)

for ax, width in zip(axes, [20, 3, 1]):
    sns.histplot(data=df, x="age", binwidth=width, color=BLUE, ax=ax)
    ax.set_title(f"Bins {width} year(s) wide")
    ax.set_xlabel("Age")
    ax.set_ylabel("")

axes[0].set_ylabel("Number of reviews")
fig.suptitle("The same age data with different bin widths", y=1.03)
plt.show()

With 20-year bins you only see that most reviewers are under 60. With 3-year bins you see the real shape: a peak in the late 30s and a long tail to the right. With 1-year bins, every age gets its own bar and random ups and downs start to dominate (there really are more 39-year-olds than 38-year-olds in this data, but that isn't a pattern worth reporting).

> **For whole-number data (ages, word counts, ratings), use a whole-number `binwidth`.** If you ask for, say, `bins=30` instead, some bins cover two whole numbers and some cover three, so the bars alternate tall and short in a pattern that isn't in the data at all.

(`fig.suptitle()` puts one title over the whole figure; `zip(axes, [20, 3, 1])` pairs each axes with its bin width, as in Lab 10.)

### Practice: Histograms

**Exercise 2: Helpful Votes (medium)**

Draw a histogram of `positive_feedback_count` with `binwidth=1`. Then draw it again, but only for reviews with **at most 20** votes (filter the DataFrame first and pass the filtered version as `data=`). In a comment, describe the shape and explain why the first chart is hard to read.

In [ ]:
# TODO: two histograms of positive_feedback_count: all reviews, and reviews with <= 20 votes

In [ ]:
# @solution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x="positive_feedback_count", binwidth=1, color=BLUE, ax=axes[0])
axes[0].set_title("All reviews")
axes[0].set_xlabel("Helpful votes")
axes[0].set_ylabel("Number of reviews")

sns.histplot(data=df[df["positive_feedback_count"] <= 20], x="positive_feedback_count",
             binwidth=1, color=BLUE, ax=axes[1])
axes[1].set_title("Reviews with at most 20 votes")
axes[1].set_xlabel("Helpful votes")
axes[1].set_ylabel("")
plt.show()
# Strongly right-skewed: almost half the reviews have 0 votes and counts fall off quickly.
# In the first chart a few reviews with 100+ votes stretch the x-axis, squeezing all the
# information into the leftmost bars.

---

### Part 4: Comparing a Number Across Groups

Does review length differ by sentiment? A **box plot** summarises a whole distribution for each group, side by side:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x="sentiment", y="n_words", order=ORDER,
            hue="sentiment", palette=SENTIMENT_COLORS, legend=False, ax=ax)
ax.set_title("Review length is similar across sentiment groups")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Words per review")
plt.show()

How to read a box:

* the **line inside the box** is the median,
* the **box** spans the middle 50% of the values (from the 25% to the 75% quartile),
* the **whiskers** reach out to the most extreme values that aren't unusually far from the box,
* any **dots** beyond the whiskers are values flagged as potential outliers.

The three boxes overlap almost completely. The medians differ by a few words (the neutral group is a little higher, matching Lab 14), but the spread within each group is far bigger than the differences between groups. A box plot shows this at a glance; a table of means hides it.

(In Seaborn, `hue="sentiment"` with `palette=SENTIMENT_COLORS` colours each box with its sentiment colour; `legend=False` skips the legend, because the x-axis already names the groups.)

### Bars of the Mean, With Uncertainty

When the question is about an **average** (average exclamation marks per sentiment), a bar chart of the mean is clearer. `sns.barplot()` computes the mean of `y` for each group, and draws a small black line showing the **95% confidence interval**: roughly, the range where the true average probably lies, given how much the data varies and how many reviews there are.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=df, x="sentiment", y="n_exclaims", order=ORDER,
            hue="sentiment", palette=SENTIMENT_COLORS, legend=False, errorbar=("ci", 95), ax=ax)
ax.set_title("Positive reviews use about twice as many exclamation marks")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Average exclamation marks per review")
plt.show()

Here the intervals are narrow and far apart: with thousands of reviews per group, the difference is clearly not a fluke. When intervals overlap heavily, be careful about claiming a difference.

### Practice: Comparing Groups

**Exercise 3: Returns by Department (medium)**

Draw a bar chart of the **share of reviews that mention a return** (`mentions_return`, which is `True`/`False`, so its mean is a share) for each `department`, in `BLUE`, with the bars ordered from highest to lowest share. (Hint: compute the order first with `groupby(...).mean().sort_values(...).index`, and pass it as `order=`.) Label it fully.

In [ ]:
# TODO: bar chart of the share of reviews mentioning a return, by department, sorted

In [ ]:
# @solution
order = df.groupby("department")["mentions_return"].mean().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=df, x="department", y="mentions_return", order=order, color=BLUE, ax=ax)
ax.set_title("Share of reviews mentioning a return, by department")
ax.set_xlabel("Department")
ax.set_ylabel("Share of reviews")
plt.show()

---

### Part 5: Relationships Between Two Numbers

A **scatter plot** draws one dot per row, with one number on each axis. With 22,000 reviews the dots pile on top of each other, so make them small and partly transparent with `alpha` (0 = invisible, 1 = solid): dense areas then show up darker.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df, x="age", y="n_words", alpha=0.08, s=12, color=BLUE, edgecolor=None, ax=ax)
ax.set_title("Reviewer age and review length are unrelated")
ax.set_xlabel("Age")
ax.set_ylabel("Words per review")
plt.show()

No slope, no shape: older reviewers don't write longer or shorter reviews. (The band along the top is the 500-character limit again.) A chart that shows "no relationship" is a perfectly good result.

### Many Pairs at Once: a Correlation Heatmap

A **correlation coefficient** summarises how strongly two numeric columns move together in a straight-line way, on a scale from **-1** to **+1**:

* **+1**: when one goes up, the other always goes up,
* **0**: no linear relationship,
* **-1**: when one goes up, the other always goes down.

`df[columns].corr()` computes it for every pair of columns, and a **heatmap** turns that table into colours:

In [ ]:
numeric_columns = ["rating", "age", "n_words", "n_exclaims", "positive_feedback_count"]
correlations = df[numeric_columns].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="RdBu", vmin=-1, vmax=1, center=0,
            linewidths=2, linecolor="white", ax=ax)
ax.set_title("Correlations between the numeric variables")
plt.show()

The colour scale is **diverging**: blue for positive correlations, red for negative, and white exactly at 0 (`center=0`), with `vmin=-1, vmax=1` fixing the ends so that colour intensity always means the same thing. `annot=True` writes the numbers in the cells, since exact values matter here.

The diagonal is always 1 (every column correlates perfectly with itself). Look at the `rating` row: its strongest correlation is with `n_exclaims` (about 0.18), and everything else is close to 0. Elsewhere, `n_words` and `positive_feedback_count` correlate at about 0.19: longer reviews collect a few more helpful votes. In words: **no single numeric feature strongly predicts the rating**, and the text features are stronger than age. That's why the next course goes further into the *content* of the text.

> Correlation only measures **straight-line** relationships, and it says nothing about cause. A correlation of 0 can hide a curved pattern, which is one more reason to look at the scatter plot too.

---

### Part 6: Shares Across Groups

Does the **sentiment mix** differ between departments? That's a question about shares within each group, and the crosstab from Lab 14 has exactly the numbers you need. A DataFrame has its own `.plot()` method (Matplotlib under the hood), and `kind="barh", stacked=True` turns each row into one bar split into its shares:

In [ ]:
shares = pd.crosstab(df["department"], df["sentiment"], normalize="index")[ORDER]
shares = shares.sort_values("negative")

fig, ax = plt.subplots(figsize=(8, 4))
shares.plot(kind="barh", stacked=True, color=[SENTIMENT_COLORS[s] for s in ORDER],
            edgecolor="white", linewidth=1.5, width=0.75, ax=ax)
ax.set_title("Trend has the largest share of negative reviews")
ax.set_xlabel("Share of reviews")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="Sentiment", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

Some details that make this chart work:

* `[ORDER]` puts the columns in a logical order (negative, neutral, positive), so the red segments all start at the same baseline (the left edge) and can be compared directly.
* Sorting by the negative share makes the ranking readable from bottom to top.
* `edgecolor="white"` leaves a thin gap between segments, so neighbouring colours don't blur together.
* The legend is moved outside the plot (`bbox_to_anchor`) so it doesn't cover the bars.

### A Heatmap of a Crosstab

For two categorical columns with more levels, a heatmap of the crosstab is often clearer than stacked bars. Here's the share of each star rating within each department. The colour scale is **sequential** (one colour, from light to dark) because the values run from low to high with no meaningful midpoint:

In [ ]:
rating_shares = pd.crosstab(df["department"], df["rating"], normalize="index")

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(rating_shares, annot=True, fmt=".0%", cmap="Blues", linewidths=2, linecolor="white", ax=ax)
ax.set_title("Share of each star rating within each department")
ax.set_xlabel("Star rating")
ax.set_ylabel("")
plt.show()

Every row is dominated by 5 stars, and the Trend row is visibly lighter on the right: fewer 5-star reviews. `fmt=".0%"` formats the numbers as percentages.

### Practice: Shares

**Exercise 4: Sentiment by Age Group (medium)**

Create the `age_group` column with `pd.cut()` as in Lab 14, then draw a stacked 100% horizontal bar chart of the sentiment shares for each age group (in their natural order, youngest at the top). Use the sentiment colours and label the chart.

In [ ]:
# TODO: add age_group, then a stacked horizontal bar chart of sentiment shares per age group

In [ ]:
# @solution
df["age_group"] = pd.cut(df["age"], bins=[17, 29, 39, 49, 59, 69, 100],
                         labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70+"])

age_shares = pd.crosstab(df["age_group"], df["sentiment"], normalize="index")[ORDER]
age_shares = age_shares.iloc[::-1]  # reverse, so the youngest group is drawn at the top

fig, ax = plt.subplots(figsize=(8, 4))
age_shares.plot(kind="barh", stacked=True, color=[SENTIMENT_COLORS[s] for s in ORDER],
                edgecolor="white", linewidth=1.5, width=0.75, ax=ax)
ax.set_title("The sentiment mix is similar across age groups")
ax.set_xlabel("Share of reviews")
ax.set_ylabel("Age group")
ax.set_xlim(0, 1)
ax.legend(title="Sentiment", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

---

### Part 7: Ordered Categories, and How Axes Can Mislead

When the x-axis has a natural **order** (age groups, months, years), a **line** or **point plot** connecting the values shows the trend. `sns.pointplot()` plots the mean for each group, joined by a line, with confidence intervals:

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[17, 29, 39, 49, 59, 69, 100],
                         labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70+"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax in axes:
    sns.pointplot(data=df, x="age_group", y="rating", color=BLUE, ax=ax)
    ax.set_xlabel("Age group")
    ax.set_ylabel("Average rating")

axes[0].set_title("Zoomed in: looks like a big difference")
axes[1].set_title("Full 1-5 scale: a small difference")
axes[1].set_ylim(1, 5)
plt.show()

Both charts show **exactly the same data**. On the left, Matplotlib zoomed the y-axis in automatically to fit the values (about 4.1 to 4.3), so a difference of 0.15 stars fills the whole height and looks dramatic. On the right, the axis covers the full rating scale, and the same difference looks as small as it really is.

Neither chart is "wrong" (a line chart, unlike a bar chart, doesn't have to start at zero), but the choice changes the message. **Always check the axis range before reading a chart, and choose yours so that the visual size of a difference matches its real importance.** Here, the right-hand chart tells the honest story: age barely matters.

The same chart type works for **time**. If your dataset has dates (Lab 13), the typical chart is the average rating or the number of reviews per month, with the months in order along the x-axis.

One more thing to avoid: charts with **two different y-axes** (for example, the number of reviews on the left axis and the average rating on the right). How the two scales line up is arbitrary, so the chart can suggest a relationship that isn't there. If you need to show two measures, draw two charts next to each other.

---

### Part 8: Charting Text

Text becomes chartable once you've counted it. Here's the Lab 08 word counter, using the regex tokenizer from Lab 10 and a stopword set, run over the whole dataset:

In [ ]:
from collections import Counter

STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "so", "of", "to", "in", "on", "at", "for",
    "with", "as", "by", "from", "up", "out", "about", "into", "than", "too", "very",
    "i", "me", "my", "it", "its", "it's", "this", "that", "these", "those", "is", "am", "are",
    "was", "were", "be", "been", "have", "has", "had", "do", "does", "did", "will", "would",
    "can", "could", "just", "also", "one", "they", "them", "you", "your", "she", "her", "we",
    "i'm", "which", "what", "when", "there", "all", "some", "more", "much", "because", "how",
    "really", "get", "got", "even", "am", "i've", "bit", "like", "dress", "top", "wear",
}

def content_tokens(text):
    tokens = [t.strip("'") for t in re.findall(r"[a-z']+", text.lower())]
    return [t for t in tokens if t and t not in STOPWORDS]

word_counts = Counter()
for text in df["review_text"]:
    word_counts.update(content_tokens(text))

top_words = pd.DataFrame(word_counts.most_common(20), columns=["word", "count"])
top_words.head()

(`t.strip("'")` removes stray apostrophes, such as the one left over from a height like `5'4`, and `if t` then drops tokens that end up empty.)

Notice that `dress`, `top`, `like`, and `wear` were added to the stopwords: they appear in so many reviews that they crowd out everything else. That's a judgement call, and it's worth stating in a report.

`pd.DataFrame(..., columns=[...])` turns the list of `(word, count)` tuples into a small DataFrame, ready to plot. For words, always use **horizontal bars**:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.barplot(data=top_words, y="word", x="count", color=BLUE, ax=ax)
ax.set_title("The 20 most frequent words in all reviews")
ax.set_xlabel("Number of occurrences")
ax.set_ylabel("")
plt.show()

### Comparing Groups Side by Side

The more interesting question is how the words differ **by sentiment**. Count the words separately for negative and positive reviews, and draw the two top-15 lists side by side. Because the groups are very different in size (about 2,400 vs 17,000 reviews), plot the **share of reviews that use each word** rather than raw counts: that puts both groups on the same scale.

In [ ]:
def top_word_shares(texts, n=15):
    # count each word at most once per review, so the result is "share of reviews using the word"
    counts = Counter()
    for text in texts:
        counts.update(set(content_tokens(text)))
    top = pd.DataFrame(counts.most_common(n), columns=["word", "reviews"])
    top["share"] = top["reviews"] / len(texts)
    return top

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, group in zip(axes, ["negative", "positive"]):
    top = top_word_shares(df.loc[df["sentiment"] == group, "review_text"])
    sns.barplot(data=top, y="word", x="share", color=SENTIMENT_COLORS[group], ax=ax)
    ax.set_title(f"{group.capitalize()} reviews")
    ax.set_xlabel("Share of reviews using the word")
    ax.set_ylabel("")

fig.suptitle("Most common words by sentiment", y=1.01)
plt.show()

Here, colour carries meaning: each panel uses its sentiment's colour, the same as every other chart in this lab.

The lists share many words (both groups talk about size, fit, and fabric), but the order and the extra words differ. `not` appears in almost half of all negative reviews, and `back` (as in *sent it back*), `ordered`, `material`, and `looked` climb the negative list, while `love`, `great`, `perfect`, `flattering`, `soft`, and `comfortable` fill the positive one. `love` even appears in the negative list, often as *"I wanted to love this"*: another reminder that single words out of context can mislead. In the next lab you'll go one step further and find the words that are most **distinctive** of each group.

### Practice: Charting Text

**Exercise 5: Title Words (medium)**

Review titles are short summaries, like `Love it!` or `Runs small`. Draw a horizontal bar chart of the 15 most common words in the **titles** of **negative** reviews, using `content_tokens()` and the negative colour. Remember that many titles are empty strings (that's fine: they simply contribute no tokens).

In [ ]:
# TODO: count content words in the titles of negative reviews, chart the top 15

In [ ]:
# @solution
title_counts = Counter()
for title in df.loc[df["sentiment"] == "negative", "title"]:
    title_counts.update(content_tokens(title))

top_title_words = pd.DataFrame(title_counts.most_common(15), columns=["word", "count"])

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=top_title_words, y="word", x="count", color=SENTIMENT_COLORS["negative"], ax=ax)
ax.set_title("Most common words in the titles of negative reviews")
ax.set_xlabel("Number of occurrences")
ax.set_ylabel("")
plt.show()

---

### Part 9: A Checklist, and Saving Your Charts

Before a chart goes into a report, check it against this list:

1. **One question per chart**, and the title states the answer when there is one.
2. **Right form for the question** (the table in Part 1).
3. **Both axes labelled**, with units where relevant ("words per review", "share of reviews").
4. **Bars start at zero**; for other charts, choose the axis range deliberately (Part 7).
5. **Sorted bars** for categories without a natural order; natural order when there is one.
6. **Colour only when it means something**, used consistently (the same colour for the same group everywhere), and colour-blind-safe. Never use colour as the *only* way to tell groups apart: add labels or a legend.
7. **One y-axis per chart.** No dual axes.
8. **No pie charts for more than a few categories**, and no 3-D effects: people compare lengths much more accurately than angles or areas.
9. **Readable at a glance**: no overlapping labels, legend out of the way.

To save a chart as an image file (for a report or presentation), call `fig.savefig()` **before** `plt.show()`:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="rating", color=BLUE, ax=ax)
ax.set_title("Most reviews give 5 stars")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of reviews")
fig.savefig("ratings.png", dpi=150, bbox_inches="tight")
plt.show()

`dpi=150` sets the resolution, and `bbox_inches="tight"` trims the empty margin. The file appears in Colab's file browser (and, like every file there, disappears when the runtime ends, unless you download it or save it to Drive).

### Practice: Putting It Together

**Exercise 6: A Chart That Answers a Question (medium)**

Answer this question with **one** well-made chart: *"Do negative reviews get more helpful votes than positive ones?"*

Think first: the votes are heavily skewed (Lab 14), so a plain bar chart of means could mislead, and a box plot would be squashed by a few reviews with 100+ votes. One good option: a bar chart of the **share of reviews with at least 3 helpful votes** for each sentiment. Make the chart, then write the answer in a comment.

In [ ]:
# TODO: one chart answering whether negative reviews get more helpful votes

In [ ]:
# @solution
df["votes_3plus"] = df["positive_feedback_count"] >= 3

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=df, x="sentiment", y="votes_3plus", order=ORDER,
            hue="sentiment", palette=SENTIMENT_COLORS, legend=False, ax=ax)
ax.set_title("Negative reviews are more often voted helpful")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Share of reviews with 3+ helpful votes")
plt.show()
# Yes: a larger share of negative (and neutral) reviews receive 3 or more helpful votes
# than positive reviews. Other shoppers seem to value detailed warnings.

---

### Recap: Your Visualization Toolkit

| Tool | What It Does | Example |
|---|---|---|
| `fig, ax = plt.subplots(figsize=(w, h))` | Create a figure with one axes | `plt.subplots(figsize=(7, 4))` |
| `fig, axes = plt.subplots(1, n)` | Several axes side by side | `axes[0]`, `axes[1]` |
| `ax.set_title/xlabel/ylabel` | Label the chart (always!) | `ax.set_xlabel("Star rating")` |
| `ax.bar` / `ax.barh` + `ax.bar_label` | Bar charts from a Series, with value labels | `ax.barh(s.index, s.values)` |
| `sns.countplot(data, x)` | Count rows per category | `sns.countplot(data=df, x="rating")` |
| `sns.histplot(data, x, bins)` | Histogram of a numeric column | `bins=40` |
| `sns.boxplot(data, x, y)` | Distribution of `y` per group | `x="sentiment", y="n_words"` |
| `sns.barplot(data, x, y)` | Mean of `y` per group, with a confidence interval | `y="n_exclaims"` |
| `sns.scatterplot(data, x, y, alpha)` | One dot per row | `alpha=0.08` |
| `df[cols].corr()` + `sns.heatmap` | Correlation matrix as colours | `cmap="RdBu", center=0` |
| `crosstab.plot(kind="barh", stacked=True)` | Stacked 100% bars of shares | `normalize="index"` first |
| `sns.pointplot(data, x, y)` | Means along an ordered x-axis | age groups, months |
| `ax.set_ylim(lo, hi)` / `ax.set_xlim` | Control the axis range | `ax.set_ylim(1, 5)` |
| `fig.savefig(path, dpi, bbox_inches)` | Save the chart as an image | `fig.savefig("chart.png", dpi=150)` |

The whole lab in one sentence: **pick the chart from the question (bar for counts, histogram for distributions, box or mean bars for groups, scatter or heatmap for relationships, stacked bars for shares), label it fully, and use colour and axis ranges honestly.**